In [14]:
%reload_ext dotenv
%dotenv ../../05_src/.secrets

In [25]:
import pandas as pd
import os
from openai import OpenAI
import requests
from OnetWebService import OnetWebService

ModuleNotFoundError: No module named 'OnetWebService'

In [16]:
soc_df = pd.read_csv("C:/Users/liane/Dropbox/liane/PhD/Data_science_course/deploying-ai/02_activities/assignment-2/SOC_codes.csv")

In [ ]:
# Read API keys from environment
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
ONET_API_KEY = os.environ.get("ONET_API_KEY")

# Initialize OpenAI client
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv("OPENAI_API_KEY")})

#Initialize O*NET web service
onet_ws = OnetWebService(ONET_API_KEY)


In [20]:
# Step 1: Extract job title from text
def extract_job_title(text: str) -> str:
    prompt = f"""
You are an assistant that reads text and extracts the job title of the person.
Return only the job title as a short string.

Text: "{text}"
"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content.strip()

In [21]:
# Step 2: Map extracted title to SOC Excel
def map_to_soc(job_title: str):
    """
    Maps extracted job title to the most granular SOC code in the CSV.
    Tries columns in order: Detailed O*NET-SOC > Detailed Occupation > Broad Occupation > Minor Group > Major Group
    Returns the SOC code and matched occupation title.
    """
    # Define the columns from most granular to least
    hierarchy_cols = [
        "Detailed O*NET-SOC",
        "Detailed Occupation",
        "Broad Occupation",
        "Minor Group",
        "Major Group"
    ]
    
    job_title_lower = job_title.lower()
    
    for col in hierarchy_cols:
        if col in soc_df.columns:
            # Case-insensitive substring match
            match = soc_df[soc_df[col].str.lower().str.contains(job_title_lower, na=False)]
            if not match.empty:
                # Return the first match: SOC code + matched occupation string
                return match.iloc[0]["SOC_code"], match.iloc[0][col]
    
    # If nothing matches, return None
    return None, None



In [16]:
# Step 3: Fetch hazards from O*NET
def get_onet_hazards(occ_code: str):
    url = f"https://services.onetcenter.org/ws/online/occupations/{occ_code}/details"
    headers = {
        "Accept": "application/json",
        "Authorization": f"Bearer {ONET_API_KEY}"
    }
    response = requests.get(url, headers=headers)
    data = response.json()
    hazards = {
        "Work Context": data.get("Work Context", {}),
        "Work Activities": data.get("Work Activities", {})
    }
    return hazards

In [17]:
# Step 4: Summarize hazards using GPT Mini 4
def summarize_hazards(job_title: str, hazard_data: dict):
    prompt = f"""
You are an occupational safety assistant.
Summarize the main hazards for the job "{job_title}" based on this O*NET data:

{hazard_data}

Return a concise summary in plain language.
"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content.strip()

In [18]:
# Step 5: Full pipeline
def extract_job_and_hazards(text: str):
    job_title = extract_job_title(text)
    occ = get_onet_occupation(job_title)
    if not occ:
        return {"job_title": job_title, "hazard_summary": "No occupation found in O*NET."}
    occ_code = occ["ocode"]
    hazards = get_onet_hazards(occ_code)
    hazard_summary = summarize_hazards(job_title, hazards)
    return {"job_title": job_title, "hazard_summary": hazard_summary}

In [20]:
# Step 6: Example usage
text_input = "Alice has been working as a teacher for several years."
result = extract_job_and_hazards(text_input)
print(result)

TypeError: Header value must be str or bytes, not <class 'NoneType'>